In [57]:
import pandas as pd
df_loaded = pd.read_json('../datasets/final_for_gen.json', orient='records')
import json 
theme_to_token = json.load(open("../pre_trained_models/ashaar/tokens/theme_tokens.json", "r"))
token_to_theme = {t:m for m,t in theme_to_token.items()}

meter_to_token = json.load(open("../pre_trained_models/ashaar/tokens/meter_tokens.json", "r"))
token_to_meter = {t:m for m,t in meter_to_token.items()}
prompat = []
completat = []
for x in range(len(df_loaded)):
    # print(df_loaded["poem qafiyeh"][x])
    try:
        theme = df_loaded["poem theme"][x]
        meter = df_loaded["poem meter"][x]
        qafiyah = df_loaded["poem qafiyeh"][x]
        prompt = f"{meter_to_token[meter]} {qafiyah} {theme_to_token[theme]}"
        # print(' '.join(df_loaded['poem verses']))
        prompat.append(prompt)
        completat.append("<|psep|>"+"<|bsep|>"+' '.join(df_loaded['poem verses'][x][:1])+"<|vsep|>"+' '.join(df_loaded['poem verses'][x][1:2])+"</|bsep|>"+"<|bsep|>"+' '.join(df_loaded['poem verses'][x][2:3])+"<|vsep|>"+' '.join(df_loaded['poem verses'][x][3:4])+"</|bsep|>"+"</|psep|>")
        # print(x)
    except:
        pass
    # encoded_input = tokenizer(prompt, return_tensors='pt')


In [58]:
completat

['<|psep|><|bsep|>كـم بـت من أسر السهاد بليلة  <|vsep|> نـاديـت فـيـهـا هـل لجـنـحك آخر</|bsep|><|bsep|>إذ قـام هـذا الصـبـح يظهر ملةً <|vsep|>  حكمت بأن ذبح الظلام الكافر</|bsep|></|psep|>',
 '<|psep|><|bsep|>عَلَيْكَ أَحَالَنِي الذِّكْرُ الجَّمِيلُ  <|vsep|> فَـجِـئْتُ ومِـنْ ثَـنَـائِكَ لي دَلِيـلُ</|bsep|><|bsep|>أَتَــيْــتُ ولَمْ أُقَــدِّمْ مِـنْ رَسُـولٍ <|vsep|>  لأَنَّ القَـلْبَ كـانَ هُوَ الرَّسُولُ</|bsep|></|psep|>',
 '<|psep|><|bsep|>أصــبــحــت فـي بـسـقـايـة مـسـلمـا <|vsep|> إلى الأعــادي لا أرى مــســلمــا</|bsep|><|bsep|>مــكــلفــاً مــا ليــس فــي طــاقـتـي<|vsep|>  مــصــفــدا مــنــتــهــرا مــرغـمـا</|bsep|></|psep|>',
 '<|psep|><|bsep|>أرئيــس الزمــان أغــفــلت أمــري <|vsep|> وتــــلذذت تـــاركـــاً لي بـــأســـر</|bsep|><|bsep|>مــا كــذا يــعـمـل الكـرام ولكـن <|vsep|>  قــد جــرى عــلى المــعـود دهـري</|bsep|></|psep|>',
 '<|psep|><|bsep|>وجـدنـا سـعـيداً منجباً خير عصبة <|vsep|>  هـمُ فـي بني أعصارهم كالمواسم</|bsep|><|bsep|>مــشــنــفــة أســمـاعـهـم بـم

In [59]:
# # Debug: inspect tokenizer, special tokens and one tokenized training example
# print("pad_token_id:", tokenizer.pad_token_id, "eos_token_id:", tokenizer.eos_token_id)
# print("special tokens ids:", {t: tokenizer.convert_tokens_to_ids(t) for t in ["<verse>","<|vsep|>","<|bsep|>","</|bsep|>","<|psep|>","</|psep|>"]})

# # show tokenization for first prompt/completion
# p = prompat[0]
# c = completat[0]
# print("prompt str:", p)
# print("completion str:", c)
# enc_p = tokenizer(p, add_special_tokens=False)["input_ids"]
# enc_c = tokenizer(c, add_special_tokens=False)["input_ids"]
# print("enc_p ids:", enc_p)
# print("enc_c ids (len):", len(enc_c))
# # build training sequence & labels exactly as your training fn
# eos = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []
# seq = enc_p + enc_c + eos
# print("seq len:", len(seq))
# # labels: -100 for prompt
# labels = [-100]*len(enc_p) + enc_c + eos
# print("labels first 30:", labels[:30])
# print("labels masked prompt count:", sum(1 for x in labels if x==-100))

In [60]:
# # Inspect tokenized dataset example
# ex = tokenized[0]
# print("input_ids:", ex["input_ids"][:60])
# print("labels   :", ex["labels"][:60])
# print("non-masked label count:", (ex["labels"] != -100).sum().item())
# # decode prompt area
# prompt_len = sum(1 for x in ex["labels"][:len(ex["labels"])] if x == -100)
# print("masked prefix tokens (should equal prompt length):", prompt_len)

In [61]:
from transformers import AutoTokenizer, AutoModelForCausalLM

from datasets import load_dataset, Dataset
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer
import torch

model_path = "../pre_trained_models/ashaar"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

# changed code
# Register special tokens BEFORE tokenizing the dataset
special_tokens = {
    "additional_special_tokens": [
        "<verse>",
        "<|vsep|>", "<|bsep|>", "</|bsep|>", "<|psep|>", "</|psep|>"
    ]
}
# ensure pad/eos exist and add special tokens
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<pad>"})
if tokenizer.eos_token is None:
    tokenizer.add_special_tokens({"eos_token": "</s>"})
tokenizer.add_special_tokens(special_tokens)
# resize model embeddings to match tokenizer
model.resize_token_embeddings(len(tokenizer))

# quick debug: confirm special tokens map to single ids
print("special tokens ids:", {t: tokenizer.convert_tokens_to_ids(t) for t in special_tokens["additional_special_tokens"]})

# build dataset from lists (prompat/completat must exist)
ds = Dataset.from_dict({"prompt": prompat, "completion": completat})
raw = ds

# use tokenizer.model_max_length if available
max_len = min(getattr(tokenizer, "model_max_length", 512), 512)

def tokenize_create_labels(examples):
    input_ids_batch = []
    attention_mask_batch = []
    labels_batch = []

    for p, c in zip(examples["prompt"], examples["completion"]):
        enc_p = tokenizer(p, add_special_tokens=False)["input_ids"]
        enc_c = tokenizer(c, add_special_tokens=False)["input_ids"]
        eos_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else None

        # reserve one position for eos if present
        reserved = 1 if eos_id is not None else 0

        # Prefer to KEEP prompt intact: truncate completion first
        max_comp = max_len - len(enc_p) - reserved
        if max_comp < 0:
            # prompt itself is too long: truncate prompt from the left to keep its tail
            enc_p = enc_p[-(max_len - reserved):]
            max_comp = max_len - len(enc_p) - reserved

        enc_c = enc_c[:max_comp] if max_comp > 0 else []

        # build final sequence: prompt + (truncated) completion + eos
        seq = enc_p + enc_c + ([eos_id] if eos_id is not None else [])
        attn = [1] * len(seq)

        # pad to max_len
        pad_len = max_len - len(seq)
        if pad_len > 0:
            seq_padded = seq + [tokenizer.pad_token_id] * pad_len
            attn_padded = attn + [0] * pad_len
        else:
            seq_padded = seq
            attn_padded = attn

        # labels: -100 for prompt tokens, real ids for completion + eos, -100 for padding
        labels = [-100] * len(enc_p) + enc_c + ([eos_id] if eos_id is not None else [])
        # safety/truncation
        labels = labels[:max_len]
        if len(labels) < max_len:
            labels = labels + [-100] * (max_len - len(labels))

        input_ids_batch.append(seq_padded)
        attention_mask_batch.append(attn_padded)
        labels_batch.append(labels)

    return {"input_ids": input_ids_batch, "attention_mask": attention_mask_batch, "labels": labels_batch}

tokenized = raw.map(tokenize_create_labels, batched=True, remove_columns=raw.column_names)
tokenized.set_format(type="torch")

# debug: show a few examples to validate masks
for i in range(3):
    ex = tokenized[i]
    print(f"example {i} input_ids len:{len(ex['input_ids'])} labels non -100 count:",
          (ex['labels'] != -100).sum().item())

# training args (lower LR if collapse happened)
training_args = TrainingArguments(
    output_dir="../models/ashaar_finetuned_8",
    overwrite_output_dir=True,
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,    # try lower LR
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    evaluation_strategy="no",
    report_to="none"
)

data_collator = None  # we already padded to max_len in tokenization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator
)

trainer.train()

Using pad_token, but it is not set yet.


special tokens ids: {'<verse>': 122, '<|vsep|>': 3, '<|bsep|>': 4, '</|bsep|>': 5, '<|psep|>': 1, '</|psep|>': 2}


Map: 100%|██████████| 3061/3061 [00:00<00:00, 5359.25 examples/s]


example 0 input_ids len:512 labels non -100 count: 110
example 1 input_ids len:512 labels non -100 count: 160
example 2 input_ids len:512 labels non -100 count: 94


/home/hussam/python_envs/complete_venv/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
  1%|▏         | 50/3820 [00:25<18:06,  3.47it/s]

...

 99%|█████████▉| 3800/3820 [15:48<00:04,  4.12it/s]

{'loss': 0.64, 'learning_rate': 1.0471204188481677e-07, 'epoch': 9.93}


100%|██████████| 3820/3820 [15:53<00:00,  4.01it/s]

{'train_runtime': 953.5735, 'train_samples_per_second': 32.1, 'train_steps_per_second': 4.006, 'train_loss': 0.8386229168058066, 'epoch': 9.98}


TrainOutput(global_step=3820, training_loss=0.8386229168058066, metrics={'train_runtime': 953.5735, 'train_samples_per_second': 32.1, 'train_steps_per_second': 4.006, 'train_loss': 0.8386229168058066, 'epoch': 9.98})

In [62]:
model.save_pretrained("../models/ashaar_finetuned_final_8")
tokenizer.save_pretrained("../models/ashaar_finetuned_final_8")


('../models/ashaar_finetuned_final_8/tokenizer_config.json',
 '../models/ashaar_finetuned_final_8/special_tokens_map.json',
 '../models/ashaar_finetuned_final_8/vocab.json',
 '../models/ashaar_finetuned_final_8/merges.txt',
 '../models/ashaar_finetuned_final_8/added_tokens.json',
 '../models/ashaar_finetuned_final_8/tokenizer.json')